# 🏥 Task 2: Command Job + MLflow Tracking

## Mục tiêu
- **Milestone 2**: Đóng gói code thành script và chạy như **Command Job** trên Azure ML
- **Milestone 3**: Tích hợp **MLflow** để theo dõi metrics, artifacts, và parameters

## Cấu trúc
```
Task2-CommandJob-MLflow/
├── Task2_CommandJob_MLflow.ipynb  ← File này
└── src/
    ├── ER_Triage_Dataset.csv
    └── train-model-mlflow.py      ← Training script
```

## Bước 1: Kiểm tra SDK

In [ ]:
pip show azure-ai-ml

## Bước 2: Kết nối Workspace

In [ ]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
    print("✅ DefaultAzureCredential OK")
except Exception as ex:
    print(f"⚠️ Chuyển sang InteractiveBrowserCredential")
    credential = InteractiveBrowserCredential()

ml_client = MLClient.from_config(credential=credential)
print(f"✅ Workspace: {ml_client.workspace_name}")

## Bước 3: Xem Training Script

Script `src/train-model-mlflow.py` thực hiện:
1. Đọc CSV và tiền xử lý (Label Encoding + MinMaxScaler)
2. Chia 70/30 train/test
3. Train Logistic Regression
4. Log vào MLflow: params, metrics (Accuracy, Recall, Precision, F1, AUC), artifacts (ROC curve, Confusion Matrix)

In [ ]:
# Xem nội dung training script
with open('src/train-model-mlflow.py', 'r') as f:
    print(f.read())

## Bước 4: Test script cục bộ (trước khi submit job)

In [ ]:
# Test script locally trước
import subprocess
result = subprocess.run(
    ["python", "src/train-model-mlflow.py",
     "--training_data", "src/ER_Triage_Dataset.csv",
     "--reg_rate", "0.01"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:500])

## Bước 5: Milestone 2 - Chạy Command Job

In [ ]:
from azure.ai.ml import command

# Cấu hình Command Job
job = command(
    code="./src",
    command="python train-model-mlflow.py --training_data ER_Triage_Dataset.csv --reg_rate 0.01",
    environment="AzureML-sklearn-0.24-ubuntu18.04-py37-cpu@latest",   # sklearn 1.5 environment
    compute="aml-cluster",
    display_name="ERTriage-command-job",
    experiment_name="ERTriage-training",
    description="Train Logistic Regression trên ER Triage Dataset",
    tags={"model_type": "LogisticRegression", "task": "classification"}
)

# Submit job
returned_job = ml_client.jobs.create_or_update(job)
aml_url = returned_job.studio_url
print(f"🚀 Command Job submitted!")
print(f"   Theo dõi tại: {aml_url}")

In [ ]:
# Chờ job hoàn thành
ml_client.jobs.stream(returned_job.name)
print("✅ Command Job hoàn thành!")

## Bước 6: Milestone 3 - Xem kết quả MLflow

In [ ]:
import mlflow

# Liệt kê tất cả experiments
print("📋 Danh sách Experiments:")
experiments = mlflow.search_experiments()
for exp in experiments:
    print(f"   - {exp.name}")

In [ ]:
# Lấy experiment ERTriage-training
experiment_name = "ERTriage-training"
exp = mlflow.get_experiment_by_name(experiment_name)
print(f"📊 Experiment: {exp.name}")
print(f"   ID: {exp.experiment_id}")

In [ ]:
# Xem tất cả runs trong experiment
import pandas as pd

runs_df = mlflow.search_runs(
    exp.experiment_id,
    order_by=["start_time DESC"]
)

# Chỉ hiển thị các cột quan trọng
cols = ['run_id', 'status', 'start_time',
        'metrics.Accuracy', 'metrics.Recall',
        'metrics.AUC', 'metrics.F1_Score',
        'params.regularization_rate']
available = [c for c in cols if c in runs_df.columns]

print("📊 Kết quả các runs:")
print(runs_df[available].to_string())

In [ ]:
# Filter: chỉ các run có AUC > 0.7 và model type là LogisticRegression
query = "metrics.AUC > 0.7 and tags.model_type = 'LogisticRegression'"
filtered_runs = mlflow.search_runs(
    exp.experiment_id,
    filter_string=query,
    order_by=["metrics.AUC DESC"]
)
print(f"\n🔍 Runs với AUC > 0.7: {len(filtered_runs)} kết quả")
if not filtered_runs.empty:
    print(filtered_runs[available].to_string())

## 📋 Kết luận Task 2

| Milestone | Hoàn thành |
|-----------|------------|
| M2: Script → Command Job | ✅ `train-model-mlflow.py` chạy trên `aml-cluster` |
| M3: MLflow Tracking | ✅ Log params, metrics (Acc/Recall/AUC/F1), artifacts (ROC, CM) |

### Metrics đã log:
- **Accuracy**: Tỷ lệ dự đoán đúng
- **Recall**: Quan trọng nhất — phát hiện đúng bệnh nặng
- **Precision**: Tỷ lệ dự đoán Regular thực sự là Regular
- **F1-Score**: Cân bằng Precision và Recall
- **AUC-ROC**: Khả năng phân biệt 2 class

### Xem trong Studio:
- **Parameters**: Jobs → Overview tab → Params
- **Metrics**: Jobs → Metrics tab
- **Artifacts**: Jobs → Images tab (ROC, CM), Outputs+Logs tab